# Additional Dataset Analysis and Results
## BIPN 162 Final Project
By Emily McClung, Naomi Ortega, and Isabel Rodriguez

We will now apply the DUNL analysis method to a Fiber Photometry dopamine spiking dataset from this paper by Seiler et al.: https://www.sciencedirect.com/science/article/pii/S0960982222001178?via%3Dihub 

### Packages set up and Data Wrangling

We will load in the data for 3 mice, each representing a different training condition
1. Mouse 1: Underwent the R160 random interval schedule of reinforcement and showed **punishment resistant reward seeking.** 
2. Mouse 2: Underwent the R160 random interval schedule of reinforcement and was **punishment sensitive.**
3. Mouse 3: Underwent the RR20 training of random ratio schedule. **Less habit formation**.

The data for each mouse contains 2 fiber photometry recording sessions. We will retrieve the dorsomedial striatum (DMS) recording signals and dorsolateral striatum (DLS) signals for each mouse.

In [1]:
!pip install pynwb h5py torch scikit-learn configmypy tensorboardX

Defaulting to user installation because normal site-packages is not writeable


In [2]:
#CLone the DUNL algorithm GitHub Repository
!git clone https://github.com/btolooshams/dunl-compneuro.git

fatal: destination path 'dunl-compneuro' already exists and is not an empty directory.


In [1]:
import numpy as np
import torch
import sklearn
import h5py
import configmypy

#Import the DUNL code
import sys
sys.path.append("dunl-compneuro/dunl")
#import dunl

#Imports for data
import os
from pynwb import NWBHDF5IO
os.makedirs("data", exist_ok=True)

In [4]:
!ls dunl-compneuro/dunl

__pycache__	     preprocess_scripts
boardfunc.py	     train_fiber_loop_acrossneurons.py
datasetloader.py     train_independentkernels_acrossneurons.py
lossfunc.py	     train_sharekernels_acrossneurons.py
model.py	     train_sharekernels_acrossneurons_groupneuralfirings.py
postprocess_scripts  utils.py


In [36]:
#Download the files to your home path by replacing with your own user- may take some time

In [ ]:
#download the dataset DANDI 000971
!pip install dandi
#download the specific files with the data we need
#RI60 PR (habit formation mimic)
!/home/emcclung/.local/bin/dandi download "https://dandiarchive.org/dandiset/000971/0.260213.1851/files?location=sub-028-392" --output-dir data

!/home/emcclung/.local/bin/dandi download "https://dandiarchive.org/dandiset/000971/0.260213.1851/files?location=sub-140-306" --output-dir data

!/home/emcclung/.local/bin/dandi download "https://dandiarchive.org/dandiset/000971/0.260213.1851/files?location=sub-100-258" --output-dir data

In [2]:
#specfic files we will look at
keep_files = {
"sub-028-392_ses-FP-PR-2020-07-24T12-31-24.nwb",
"sub-028-392_ses-FP-PR-2020-07-09T13-01-26.nwb",
"sub-140-306_ses-FP-PS-2019-08-09T12-10-58_behavior.nwb",
"sub-140-306_ses-FP-PS-2019-09-03T10-15-44_behavior.nwb",
"sub-100-258_ses-FP-RR20-2019-05-09T13-32-40_behavior.nwb",
"sub-100-258_ses-FP-RR20-2019-04-23T12-25-00_behavior.nwb"
}

data_dir = "data"
#delete unused files 
for root, dirs, files in os.walk(data_dir):
    for f in files:
        if f.endswith(".nwb") and f not in keep_files:
            path = os.path.join(root, f)
            os.remove(path)
    print("Deleted unused files")

Deleted unused files
Deleted unused files
Deleted unused files
Deleted unused files
Deleted unused files


In [3]:
#Group files by subject
dir_028 = "data/sub-028-392"
files_028 = [f for f in os.listdir(dir_028) if f.endswith(".nwb")]

dir_140 = "data/sub-140-306"
files_140 = [f for f in os.listdir(dir_140) if f.endswith(".nwb")]

dir_100 = "data/sub-028-258"
dir_100 = "data/sub-100-258"
files_100 = [f for f in os.listdir(dir_100) if f.endswith(".nwb")]

print(files_028)
print(files_140)
print(files_100)

['sub-028-392_ses-FP-PR-2020-07-09T13-01-26.nwb', 'sub-028-392_ses-FP-PR-2020-07-24T12-31-24.nwb']
['sub-140-306_ses-FP-PS-2019-08-09T12-10-58_behavior.nwb', 'sub-140-306_ses-FP-PS-2019-09-03T10-15-44_behavior.nwb']
['sub-100-258_ses-FP-RR20-2019-04-23T12-25-00_behavior.nwb', 'sub-100-258_ses-FP-RR20-2019-05-09T13-32-40_behavior.nwb']


In [4]:
#extracts dms_signal, dls_signal, and timestamps from files
def load_fp_signal(file_path):

    with NWBHDF5IO(file_path, "r") as io:
        nwb = io.read()

        fp_series = nwb.acquisition['fiber_photometry_response_series']
        raw_signal = fp_series.data[:]

        rate = fp_series.rate
        start = fp_series.starting_time
        timestamps = start + (np.arange(len(raw_signal)) / rate)

        dms_signal = raw_signal[:,0]
        dls_signal = raw_signal[:,2]

    return dms_signal, dls_signal, timestamps

### Changes made to configurations in the dunl-compneuro repo:
We made these changes to the configurations of the repo in order to use their code with new data.

1. In `config/dopamine_fiberphotometry_saramatias_uchida_config_1window_1kernel.yaml`
changed **from** `data_folder:  "../data/dopamine-spiking-eshel-uchida"`
**to** `data_folder: "data/dls_sessions.pt"`

2. In `dunl/train_fiber_loop_acrossneurons.py ` First, inside `init_params()`
 **set** `default="dunl-compneuro/config"`
to follow the correct path. Second, confirm that `default="./dopamine_calcium_saramatias_uchida_config.yaml"` is the line that is commented out, and all the other default lines are. Since 000971 data is FP, it uses the calcium setting. (recreation used spiking setting)

3. In `config/dopamine_fiberphotometry_saramatias_uchida_config_1window_1kernel.yaml`, change number_of_window from 19 to 50 to make sure the reshaping works with the size of our data.

4. In `dopamine_fiberphotometry_saramatias_uchida_config_1window_1kernel.yaml` change train_num_epochs: from **1000** to **100**. This is key for reducing training time from 1 hour to under 10 minutes.

5. In `dunl/train_fiber_loop_acrossneurons.py`, in the init_params() change `dataset.y = dataset.y[0]` to `dataset.y = dataset.y[:, 0, :]` so that both of the 2 trials for each DUNL run get input, instead of just 1. **directly below** that line, added `dataset.y = dataset.y[:, :15000]` to force the signal cropping to 15000 because training was getting hung up at reshaping for dunl trial 6 due to long signal size.


7. In `dunl/datasetloader.py`
**added** `'import numpy'`
inside `init()`**added** `with torch.serialization.safe_globals([numpy.core.multiarray._reconstruct]):
            data = torch.load(data_path, weights_only=False)`. Note: this one is necessary for newer PyTorch versions only (for Naomi it didn't work with this change)

In [20]:
#!rm-r folderpath/foldername
!rm -r ../971_dunl_results/fiber_numwindow50_neuron0_kernellength30_1kernels_1000unroll_2026_03_11_06_39_06

rm: cannot remove '../971_dunl_results/fiber_numwindow50_neuron0_kernellength30_1kernels_1000unroll_2026_03_11_06_39_06': No such file or directory


# Data Analysis

## DUNL Runs
We perform DUNL on the 3 mice. For each mouse, DUNL is run on both the DMS data and the DLS data, for a total of 6 total DUNL runs.

## Mouse 1: A Punishment Resistant Mouse trained with random interval schedule of reinforcement (R160)

In [32]:
#initalize lists
dms_sessions = []
dls_sessions = []

#loop through files and load signals
for f in files_028:
    file_path = os.path.join(dir_028, f)
    dms, dls, timestamps = load_fp_signal(file_path)
    dms_sessions.append(dms)
    dls_sessions.append(dls)

#convert lists to object arrays
dms_sessions = np.array(dms_sessions, dtype=object)
dls_sessions = np.array(dls_sessions, dtype=object)

#find the shortest session length to ensure uniform matrix dimensions
min_len = min(len(s) for s in dms_sessions)

#trim sessions to the minimum length and add the 'neuron' dimension (axis 1)
#shape: (num_trials, 1, trial_length)
dms_matrix = np.array([s[:min_len] for s in dms_sessions])[:, np.newaxis, :]
dls_matrix = np.array([s[:min_len] for s in dls_sessions])[:, np.newaxis, :]

#define dimensions for metadata
num_trials, num_neurons, trial_length = dls_matrix.shape

#fefine a helper to package and save as PyTorch tensors immediately
def save_as_tensor_dict(matrix, filename):
    tensor_dict = {
        "y": torch.as_tensor(matrix, dtype=torch.float32),
        "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
        "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
        "type": torch.zeros(num_trials, dtype=torch.int64)
    }
    torch.save(tensor_dict, filename)
    return tensor_dict

#save processed data
dms_sessions_fixed = save_as_tensor_dict(dms_matrix, "data/dms_sessions.pt")
dls_sessions_fixed = save_as_tensor_dict(dls_matrix, "data/dls_sessions.pt")

#verify types and shapes
for key, value in dls_sessions_fixed.items():
    print(f"DLS {key}: {type(value)} | Shape: {value.shape}")

DLS y: <class 'torch.Tensor'> | Shape: torch.Size([2, 1, 1124608])
DLS x: <class 'torch.Tensor'> | Shape: torch.Size([2, 1, 1])
DLS a: <class 'torch.Tensor'> | Shape: torch.Size([2, 1, 1])
DLS type: <class 'torch.Tensor'> | Shape: torch.Size([2])


## Brain area: DLS
DUNL 1/6

In [33]:
# For each run of DUNL, the input is 2 sessions.
#keep first 15000 timepoints of each session
original_data = torch.as_tensor(dls_matrix, dtype=torch.float32)

y_small = original_data[:, :, :15000]

num_trials = y_small.shape[0]
num_neurons = y_small.shape[1]

dls_test_set = {
    "y": y_small,
    "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "type": torch.zeros(num_trials, dtype=torch.int64)
}

torch.save(dls_test_set, "data/dls_sessions.pt")

print("New shape:", y_small.shape)
print("2 trials, 1 neuron, 15000 time points")
print("Points per window:", y_small.shape[2] // 50)

New shape: torch.Size([2, 1, 15000])
2 trials, 1 neuron, 15000 time points
Points per window: 300


In [34]:
!python dunl-compneuro/dunl/train_fiber_loop_acrossneurons.py 

init parameters.
neuron_index 0
Train DUNL on neural data (independent kernels for each neuron).
device is cuda:0
Exp: fiber
create dataset and dataloader.
There 1 dataset in the folder.
data/dls_sessions.pt
x is shared among neurons. It is a function of trials, and number of kernels!
create board.
x is shared among neurons. It is a function of trials, and number of kernels!
This script has a hardcoded step for dataset processing.
these are hardcoded now! should be moved into pre-processing step.
create model.
create optimizer and scheduler for training.
start training.
100%|█████████████████████████████████████████| 100/100 [03:54<00:00,  2.35s/it]


### Brain area: DMS
DUNL 2/6

In [8]:
# For each run of DUNL, the input is 2 sessions.
#keep first 15000 timepoints of each session
original_data = torch.as_tensor(dms_matrix, dtype=torch.float32)

y_small = original_data[:, :, :15000]

num_trials = y_small.shape[0]
num_neurons = y_small.shape[1]

dms_test_set = {
    "y": y_small,
    "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "type": torch.zeros(num_trials, dtype=torch.int64)
}

torch.save(dms_test_set, "data/dms_sessions.pt")

print("New shape:", y_small.shape)
print("2 trials, 1 neuron, 15000 time points")
print("Points per window:", y_small.shape[2] // 50)

New shape: torch.Size([2, 1, 15000])
2 trials, 1 neuron, 15000 time points
Points per window: 300


In [9]:
!python dunl-compneuro/dunl/train_fiber_loop_acrossneurons.py 

init parameters.
neuron_index 0
Train DUNL on neural data (independent kernels for each neuron).
device is cuda:0
Exp: fiber
create dataset and dataloader.
There 1 dataset in the folder.
data/dls_sessions.pt
x is shared among neurons. It is a function of trials, and number of kernels!
create board.
x is shared among neurons. It is a function of trials, and number of kernels!
This script has a hardcoded step for dataset processing.
these are hardcoded now! should be moved into pre-processing step.
create model.
create optimizer and scheduler for training.
start training.
100%|█████████████████████████████████████████| 100/100 [03:55<00:00,  2.36s/it]


## Mouse 2: A Punishment Sensitive R160 Mouse

In [17]:
#initalize lists
dms_sessions = []
dls_sessions = []

#loop through files and load signals
for f in files_140:
    file_path = os.path.join(dir_140, f)
    dms, dls, timestamps = load_fp_signal(file_path)
    dms_sessions.append(dms)
    dls_sessions.append(dls)

#convert lists to object arrays
dms_sessions = np.array(dms_sessions, dtype=object)
dls_sessions = np.array(dls_sessions, dtype=object)

#find the shortest session length to ensure uniform matrix dimensions
min_len = min(len(s) for s in dms_sessions)

#trim sessions to the minimum length and add the 'neuron' dimension (axis 1)
#shape: (num_trials, 1, trial_length)
dms_matrix = np.array([s[:min_len] for s in dms_sessions])[:, np.newaxis, :]
dls_matrix = np.array([s[:min_len] for s in dls_sessions])[:, np.newaxis, :]

#define dimensions for metadata
num_trials, num_neurons, trial_length = dls_matrix.shape

#fefine a helper to package and save as PyTorch tensors immediately
def save_as_tensor_dict(matrix, filename):
    tensor_dict = {
        "y": torch.as_tensor(matrix, dtype=torch.float32),
        "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
        "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
        "type": torch.zeros(num_trials, dtype=torch.int64)
    }
    torch.save(tensor_dict, filename)
    return tensor_dict

#save processed data
dms_sessions_fixed = save_as_tensor_dict(dms_matrix, "data/dms_sessions.pt")
dls_sessions_fixed = save_as_tensor_dict(dls_matrix, "data/dls_sessions.pt")

#verify types and shapes
for key, value in dls_sessions_fixed.items():
    print(f"DLS {key}: {type(value)} | Shape: {value.shape}")

DLS y: <class 'torch.Tensor'> | Shape: torch.Size([2, 1, 3743488])
DLS x: <class 'torch.Tensor'> | Shape: torch.Size([2, 1, 1])
DLS a: <class 'torch.Tensor'> | Shape: torch.Size([2, 1, 1])
DLS type: <class 'torch.Tensor'> | Shape: torch.Size([2])


### Brain area: DLS
DUNL 3/6

In [18]:
# For each run of DUNL, the input is 2 sessions.
#keep first 15000 timepoints of each session
original_data = torch.as_tensor(dls_matrix, dtype=torch.float32)

y_small = original_data[:, :, :15000]

num_trials = y_small.shape[0]
num_neurons = y_small.shape[1]

dls_test_set = {
    "y": y_small,
    "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "type": torch.zeros(num_trials, dtype=torch.int64)
}

torch.save(dls_test_set, "data/dls_sessions.pt")

print("New shape:", y_small.shape)
print("2 trials, 1 neuron, 15000 time points")
print("Points per window:", y_small.shape[2] // 50)

New shape: torch.Size([2, 1, 15000])
2 trials, 1 neuron, 15000 time points
Points per window: 300


In [19]:
!python dunl-compneuro/dunl/train_fiber_loop_acrossneurons.py 

init parameters.
neuron_index 0
Train DUNL on neural data (independent kernels for each neuron).
device is cuda:0
Exp: fiber
create dataset and dataloader.
There 1 dataset in the folder.
data/dls_sessions.pt
x is shared among neurons. It is a function of trials, and number of kernels!
create board.
x is shared among neurons. It is a function of trials, and number of kernels!
This script has a hardcoded step for dataset processing.
these are hardcoded now! should be moved into pre-processing step.
create model.
create optimizer and scheduler for training.
start training.
100%|█████████████████████████████████████████| 100/100 [03:54<00:00,  2.35s/it]


### Brain area: DMS
DUNL 4/6

In [22]:
# For each run of DUNL, the input is 2 sessions.
#keep first 15000 timepoints of each session
original_data = torch.as_tensor(dms_matrix, dtype=torch.float32)

y_small = original_data[:, :, :15000]

num_trials = y_small.shape[0]
num_neurons = y_small.shape[1]

dms_test_set = {
    "y": y_small,
    "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "type": torch.zeros(num_trials, dtype=torch.int64)
}

torch.save(dms_test_set, "data/dms_sessions.pt")

print("New shape:", y_small.shape)
print("2 trials, 1 neuron, 15000 time points")
print("Points per window:", y_small.shape[2] // 50)

New shape: torch.Size([2, 1, 15000])
2 trials, 1 neuron, 15000 time points
Points per window: 300


In [23]:
!python dunl-compneuro/dunl/train_fiber_loop_acrossneurons.py 

init parameters.
neuron_index 0
Train DUNL on neural data (independent kernels for each neuron).
device is cuda:0
Exp: fiber
create dataset and dataloader.
There 1 dataset in the folder.
data/dls_sessions.pt
x is shared among neurons. It is a function of trials, and number of kernels!
create board.
x is shared among neurons. It is a function of trials, and number of kernels!
This script has a hardcoded step for dataset processing.
these are hardcoded now! should be moved into pre-processing step.
create model.
create optimizer and scheduler for training.
start training.
100%|█████████████████████████████████████████| 100/100 [03:56<00:00,  2.37s/it]


## Mouse 3: RR20 Mouse

In [5]:
#initalize lists
dms_sessions = []
dls_sessions = []

#loop through files and load signals
for f in files_100:
    file_path = os.path.join(dir_100, f)
    dms, dls, timestamps = load_fp_signal(file_path)
    dms_sessions.append(dms)
    dls_sessions.append(dls)

#convert lists to object arrays
dms_sessions = np.array(dms_sessions, dtype=object)
dls_sessions = np.array(dls_sessions, dtype=object)

#find the shortest session length to ensure uniform matrix dimensions
min_len = min(len(s) for s in dms_sessions)

#trim sessions to the minimum length and add the 'neuron' dimension (axis 1)
#shape: (num_trials, 1, trial_length)
dms_matrix = np.array([s[:min_len] for s in dms_sessions])[:, np.newaxis, :]
dls_matrix = np.array([s[:min_len] for s in dls_sessions])[:, np.newaxis, :]

#define dimensions for metadata
num_trials, num_neurons, trial_length = dls_matrix.shape

#fefine a helper to package and save as PyTorch tensors immediately
def save_as_tensor_dict(matrix, filename):
    tensor_dict = {
        "y": torch.as_tensor(matrix, dtype=torch.float32),
        "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
        "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
        "type": torch.zeros(num_trials, dtype=torch.int64)
    }
    torch.save(tensor_dict, filename)
    return tensor_dict

#save processed data
dms_sessions_fixed = save_as_tensor_dict(dms_matrix, "data/dms_sessions.pt")
dls_sessions_fixed = save_as_tensor_dict(dls_matrix, "data/dls_sessions.pt")

#verify types and shapes
for key, value in dls_sessions_fixed.items():
    print(f"DLS {key}: {type(value)} | Shape: {value.shape}")

/home/n2ortega/.local/lib/python3.11/site-packages/hdmf/spec/namespace.py:484: UserWarning: Schema conflict(s) detected in namespace 'ndx-fiber-photometry': 
 ndx-fiber-photometry defines OpticalFiber.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines ExcitationSource.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines Photodetector.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines DichroicMirror.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines BandOpticalFilter.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel.
ndx-fiber-photometry defines EdgeOpticalFilter.model as an attribute (dtype: text) while the core schema defines it as a link to DeviceModel. 
T

DLS y: <class 'torch.Tensor'> | Shape: torch.Size([2, 1, 3713536])
DLS x: <class 'torch.Tensor'> | Shape: torch.Size([2, 1, 1])
DLS a: <class 'torch.Tensor'> | Shape: torch.Size([2, 1, 1])
DLS type: <class 'torch.Tensor'> | Shape: torch.Size([2])


### Brain area: DLS
DUNL 5/6

In [55]:
# For each run of DUNL, the input is 2 sessions.
#keep first 15000 timepoints of each session
original_data = torch.as_tensor(dls_matrix, dtype=torch.float32)

y_small = original_data[:, :, :15000]

num_trials = y_small.shape[0]
num_neurons = y_small.shape[1]

dls_test_set = {
    "y": y_small,
    "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "type": torch.zeros(num_trials, dtype=torch.int64)
}

torch.save(dls_test_set, "data/dls_sessions.pt")

print("New shape:", y_small.shape)
print("2 trials, 1 neuron, 15000 time points")
print("Points per window:", y_small.shape[2] // 50)

New shape: torch.Size([2, 1, 15000])
2 trials, 1 neuron, 15000 time points
Points per window: 300


In [27]:
!python dunl-compneuro/dunl/train_fiber_loop_acrossneurons.py 

init parameters.
neuron_index 0
Train DUNL on neural data (independent kernels for each neuron).
device is cuda:0
Exp: fiber
create dataset and dataloader.
There 1 dataset in the folder.
data/dls_sessions.pt
x is shared among neurons. It is a function of trials, and number of kernels!
create board.
x is shared among neurons. It is a function of trials, and number of kernels!
This script has a hardcoded step for dataset processing.
these are hardcoded now! should be moved into pre-processing step.
create model.
create optimizer and scheduler for training.
start training.
100%|█████████████████████████████████████████| 100/100 [03:55<00:00,  2.36s/it]


### Brain area: DMS
DUNL 6/6

In [11]:
# For each run of DUNL, the input is 2 sessions.
#keep first 15000 timepoints of each session
original_data = torch.as_tensor(dms_matrix, dtype=torch.float32)
print(dms_matrix.shape)
y_small = original_data[:, :, :15000]
print(y_small.shape)
num_trials = y_small.shape[0]
num_neurons = y_small.shape[1]

dms_test_set = {
    "y": y_small,
    "x": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "a": torch.zeros((num_trials, num_neurons, 1), dtype=torch.float32),
    "type": torch.zeros(num_trials, dtype=torch.int64)
}

torch.save(dms_test_set, "data/dms_sessions.pt")

print("New shape:", y_small.shape)
print("2 trials, 1 neuron, 15000 time points")
print("Points per window:", y_small.shape[2] // 50)

(2, 1, 3713536)
torch.Size([2, 1, 15000])
New shape: torch.Size([2, 1, 15000])
2 trials, 1 neuron, 15000 time points
Points per window: 300


In [12]:
!python dunl-compneuro/dunl/train_fiber_loop_acrossneurons.py 

init parameters.
neuron_index 0
Train DUNL on neural data (independent kernels for each neuron).
device is cuda:0
Exp: fiber
create dataset and dataloader.
There 1 dataset in the folder.
data/dls_sessions.pt
x is shared among neurons. It is a function of trials, and number of kernels!
create board.
x is shared among neurons. It is a function of trials, and number of kernels!
This script has a hardcoded step for dataset processing.
these are hardcoded now! should be moved into pre-processing step.
create model.
create optimizer and scheduler for training.
start training.
100%|█████████████████████████████████████████| 100/100 [03:55<00:00,  2.35s/it]
